# Electrical Grid Stability Classification
## Notebook 01: Exploratory Data Analysis & System Diagnostics

This notebook provides a thorough empirical and exploratory analysis of the **Electrical Grid Stability Simulated Data Set** (UCI Machine Learning Repository ID: 471).

### Domain Background
The dataset models a decentralized four-node electrical power network consisting of:
- 1 power generator node ($p_1 > 0$)
- 3 electrical power consumer nodes ($p_2, p_3, p_4 < 0$)

For each participant node $i \in \{1, 2, 3, 4\}$:
- $\tau_i$: Reaction time parameter (seconds required to adjust generation or consumption to price fluctuations).
- $p_i$: Nominal power produced or consumed (with physical balance $\sum_{i=1}^4 p_i = 0$).
- $g_i$: Price elasticity coefficient (willingness to adapt electricity consumption/generation to market prices).

The target variable `stabf` identifies dynamic synchronization stability:
- **`stable` (0)**: System successfully returns to synchronous equilibrium.
- **`unstable` (1)**: Divergence or loss of synchronization.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data
from src.validation import validate_raw_dataset
from config.config import TAU_FEATURES, POWER_FEATURES, ELASTICITY_FEATURES, INPUT_FEATURES

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

### 1. Ingest Raw Dataset & Schema Verification

In [ ]:
df_raw = load_raw_data()
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

### 2. Validation & Zero-Leakage Separation
Notice the continuous variable `stab`. It is the maximal real part of the characteristic differential equation roots. Because `stabf` is derived directly as `stab <= 0 -> stable`, retaining `stab` in the feature set would constitute fatal target leakage. We validate that `stab` is discarded and target labels are encoded as `0 = Stable` and `1 = Unstable`.

In [ ]:
X, y, report = validate_raw_dataset(df_raw)
print("Validation Summary:")
print(f"- Input Feature Matrix: {X.shape}")
print(f"- Leakage column removed: {report['dropped_leakage_col']}")
print(f"- Missing values detected: {sum(report['missing_values'].values())}")
print(f"- Duplicate rows: {report['duplicate_rows']}")

pd.DataFrame(report['target_summary'])

### 3. Target Distribution
Approximately 63.8% of simulations exhibit instability and 36.2% exhibit stability.

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x=y.map({0: 'Stable (0)', 1: 'Unstable (1)'}), palette=['#2ca02c', '#d62728'])
plt.title("Class Distribution: Electrical Grid Stability")
plt.xlabel("Operational State")
plt.ylabel("Observation Count")
for p in ax.patches:
    ax.annotate(f"{p.get_height()} ({p.get_height()/len(y)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()

### 4. Feature Distributions by Parameter Group
We examine the marginal distributions of reaction times ($\tau$), nominal powers ($p$), and elasticities ($g$).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
df_plot = X.copy()
df_plot['Stability'] = y.map({0: 'Stable', 1: 'Unstable'})

for idx, col in enumerate(INPUT_FEATURES):
    ax = axes[idx // 4, idx % 4]
    sns.histplot(data=df_plot, x=col, hue='Stability', kde=True, ax=ax, palette={'Stable': '#2ca02c', 'Unstable': '#d62728'}, alpha=0.4)
    ax.set_title(col, fontsize=12, fontweight='bold')

plt.suptitle("Feature Distributions Conditioned on Grid Stability", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 5. Boxplot Comparisons by Class
Examining parameter ranges associated with stability vs instability.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, cols) in enumerate([('Reaction Time (tau)', TAU_FEATURES), ('Power (p)', POWER_FEATURES), ('Elasticity (g)', ELASTICITY_FEATURES)]):
    melted = pd.melt(df_plot, id_vars=['Stability'], value_vars=cols, var_name='Parameter', value_name='Value')
    sns.boxplot(data=melted, x='Parameter', y='Value', hue='Stability', palette={'Stable': '#2ca02c', 'Unstable': '#d62728'}, ax=axes[idx])
    axes[idx].set_title(f"{name} by Stability", fontsize=13)

plt.tight_layout()
plt.show()

### 6. Correlation Analysis
Analyzing pairwise correlations among electrical parameters.

In [ ]:
plt.figure(figsize=(10, 8))
corr = X.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix of Input Features", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

### 7. Physical Interpretation & Non-Causality Statement
> **Scientific Caveat**: The observed statistical differences indicate parameter regimes associated with higher probability of stability within the simulation dataset. These relationships do not prove direct physical causality in operational power systems.